# Life Cycle Inventory: 3D-Printed Biopolymer — Master Notebook

**Project:** RAW Project — 3D-Printed Biopolymer Panels  
**Database:** `lca_database_3DPrintedBiopol`  
**Ecoinvent version:** 3.12, cut-off system model  
**Date:** 2026  

## Purpose

This notebook is the single authoritative source for the foreground LCA database used in this study. It builds the database from scratch and documents every modelling decision: system boundary, functional unit, activity structure, exchange selection, and numerical assumptions.

Running this notebook in full will delete any existing version of `lca_database_3DPrintedBiopol` and recreate it. All results in the associated paper are reproducible from this notebook together with ecoinvent 3.12 cut-off (not redistributed; see prerequisites below).

## Prerequisites

- Python environment with `brightway2` or `bw2data` / `bw2io` / `bw2calc` (≥ 2.5)
- ecoinvent 3.12, cut-off system model, imported into a Brightway project named `biopol_lca`
- See `README.md` for setup instructions

## Structure

| Section | Content |
|---|---|
| 0 | Setup, version pinning, safety export |
| 1 | System description, scope, and functional unit |
| 2 | System boundary diagram |
| 3 | Background database and shared modelling conventions |
| 4 | Helper functions |
| 5 | Database initialisation |
| 6 | Filler production activities |
| 7 | Binder production chain |
| 8 | Mix preparation |
| 9 | 3D printing |
| 10 | Drying |
| 11 | Kiln baking |
| 12 | Avoided burden: nitrogen fixation |
| 13 | Full database verification |

---

## 0. Setup, version pinning, and safety export

In [11]:
import bw2data as bd
import bw2io as bi
import bw2calc as bc
import importlib.metadata

# ── Version pinning ───────────────────────────────────────────────────────────
# Print environment versions so the computational context is documented
# in the notebook output.
for pkg in ["bw2data", "bw2io", "bw2calc"]:
    print(f"  {pkg}: {importlib.metadata.version(pkg)}")
print()

# ── Connect to project ────────────────────────────────────────────────────────
bd.projects.set_current("biopol_lca")
ei = bd.Database("ecoinvent-3.12-cutoff")
print(f"  ecoinvent: {len(ei)} datasets")

  bw2data: 4.4.2
  bw2io: 0.9.6
  bw2calc: 2.0.1

  ecoinvent: 26533 datasets


In [12]:
# ── Safety export ─────────────────────────────────────────────────────────────
# Export the current database to Excel before deleting it.
# Skip if the database does not yet exist.
DB_NAME = "lca_database_3DPrintedBiopol"

if DB_NAME in bd.databases:
    export_path = bi.export.write_lci_excel(DB_NAME, dirpath="/Users/tsuitpy/Library/CloudStorage/OneDrive-UniversiteitLeiden/01_Projects/RAW/lca_for_adaptive_factories/notebooks/260622_biopolLCA_final")
    print(f"Existing database exported to: {export_path}")
else:
    print("No existing database found — skipping export.")

Existing database exported to: /Users/tsuitpy/Library/CloudStorage/OneDrive-UniversiteitLeiden/01_Projects/RAW/lca_for_adaptive_factories/notebooks/260622_biopolLCA_final/lci-lca_database_3DPrintedBiopol.xlsx


## 1. System description, scope, and functional unit

### Product system

This study presents a life cycle assessment (LCA) of a bio-based composite material for interior construction applications, developed within the RAW Project (Residual Agricultural Waste). The material consists of a pea protein binder combined with bio-based and waste-derived fillers, processed into a 3D-printed panel using robotic extrusion. The reference recipe (Recipe 1, Gitsouli protocol, DTU/CITA) uses hemp dust as the active filler, with pea protein isolate, glycerol, water, and xanthan gum forming the binder paste.

### Functional unit

**1 kg of 3D-printed biopolymer material, dried and baked, ready for use.**

### System boundary

**Included:**
- Filler production (collection, drying, milling) for all six filler materials
- Binder production (pea protein isolation via AEIEP process, binder paste preparation)
- Mix preparation (filler–binder homogenisation)
- 3D printing (robotic extrusion)
- Drying (dehumidifying chamber, 2 weeks)
- Kiln baking (150°C, 45 minutes)
- Use phase *(further work)*
- End of life *(further work)*

**Excluded:**
- Capital equipment beyond machine wear (buildings, infrastructure, utilities)
- Human labour

### Attributional vs. consequential

This is an attributional LCA using the cut-off system model (ecoinvent 3.12, cut-off). Under the cut-off approach, waste and by-product flows at the point of generation carry no upstream burden; downstream users of those flows bear no upstream impacts. This is consistent with the treatment of zero-burden raw materials in this study (seagrass wrack, hemp dust, cellulose reject fines).

### Recipe reference

All mass fractions throughout the database are derived from Recipe 1 (Gitsouli protocol, DTU/CITA). Recipe 1 defines the reference composition: 80.1% binder fraction (pea protein isolate, glycerol, water, xanthan gum) and 19.9% hemp dust filler. Alternative filler materials (bark flour, cellulose, cotton, seagrass, wood flour) are retained as zero-amount placeholder inputs in the mix preparation activity to support future optimisation runs with alternative recipes without modifying the database structure.

## 2. System boundary diagram

The diagram below shows the foreground activity structure and the flow of materials through the product system. Arrows indicate technosphere inputs. Background ecoinvent datasets (electricity, transport, chemicals) are omitted for clarity. Use phase and end of life are indicated as further work.

```mermaid
flowchart TD
    %% Filler production
    SG[seagrass filler production]
    BK[bark flour filler production]
    CE[cellulose filler production]
    CO[cotton filler production]
    HD[hemp dust filler production]
    WF[wood flour filler production]

    %% Binder chain
    AE[AEIEP pea protein isolate production]
    BI[pea protein binder production]
    AV[avoided burden: nitrogen fixation]

    %% Downstream
    MX[mix preparation]
    PR[3D printing]
    DR[drying]
    KI[kiln baking]

    %% Further work
    UP[use phase]
    EL[end of life]

    AE --> BI
    AE -.-> AV

    SG --> MX
    BK --> MX
    CE --> MX
    CO --> MX
    HD --> MX
    WF --> MX
    BI --> MX

    MX --> PR
    PR --> DR
    DR --> KI
    KI -.-> UP
    UP -.-> EL

    style KI fill:#c8e6c9
    style AV fill:#fff9c4
    style UP fill:#f5f5f5,stroke:#bdbdbd,stroke-dasharray: 5 5,color:#9e9e9e
    style EL fill:#f5f5f5,stroke:#bdbdbd,stroke-dasharray: 5 5,color:#9e9e9e
```

> **Green:** reference product (functional unit output). **Yellow:** avoided burden (negative contribution). **Dashed arrows:** avoided burden linkage and further work boundaries. **Grey dashed boxes:** use phase and end of life — not yet modelled. Filler inputs to `mix preparation` are placeholders at 0 kg in Recipe 1, except hemp dust (0.199 kg).

## 3. Background database and shared modelling conventions

### Background database

All background processes are drawn from **ecoinvent 3.12, cut-off system model**. The cut-off approach assigns no burden or credit to waste and by-product flows at the point of generation; downstream users of those flows bear no upstream impacts. This is consistent with the treatment of bio-based raw materials in this study (seagrass wrack, hemp dust, bark chips, sawdust).

### Shared conventions applied throughout the database

The following choices apply to every activity unless stated otherwise in a section below.

| Convention | Choice | Rationale |
|---|---|---|
| Electricity | `market for electricity, medium voltage \| DE` | Study is conducted in a European research context; DE grid used as representative |
| Transport | `market for transport, freight, lorry, unspecified \| RER` | Generic European road freight |
| Transport distance | 50 km | Default assumption for all material inputs without a known supply chain; used consistently across fillers |
| Machine wear | `market for industrial machine, heavy, unspecified \| RER` | Generic proxy for lab-scale cutting and processing equipment |
| Machine wear amount | 0.01 kg/kg output | Lab-scale estimate based on Retsch SM2000 cutting mill; revise to 1e-4 to 1e-6 for industrial-scale LCA |
| Activity location | GLO for material production; RER for processing | Reflects global material sourcing and European processing |

---

## 4. Helper functions

In [13]:
def find_activity(db, name):
    """Find a single foreground activity by exact name."""
    results = [a for a in db if a['name'] == name]
    if len(results) == 0:
        raise ValueError(f"Activity not found: '{name}'")
    if len(results) > 1:
        raise ValueError(f"Multiple activities found for: '{name}'")
    return results[0]


def find_ei(name, location, ref_product=None):
    """Look up an ecoinvent activity by exact name and location."""
    results = [
        a for a in ei
        if a['name'] == name
        and a['location'] == location
        and (ref_product is None or a.get('reference product') == ref_product)
    ]
    if len(results) == 0:
        raise ValueError(f"Ecoinvent dataset not found: '{name}' | {location}")
    if len(results) > 1:
        print(f"  Warning: multiple matches for '{name}' | {location} — using first")
    return results[0]


def search_ei(keyword, max_results=20):
    """
    Broad keyword search across ecoinvent. Used to verify dataset names
    before hard-coding them in find_ei().
    """
    keyword_lower = keyword.lower()
    results = [a for a in ei if keyword_lower in a['name'].lower()]
    print(f"Found {len(results)} dataset(s) matching '{keyword}':")
    for a in results[:max_results]:
        print(f"  name:     {a['name']}")
        print(f"  location: {a['location']}")
        print(f"  ref prod: {a.get('reference product', '—')}")
        print(f"  unit:     {a.get('unit', '—')}")
        print()
    if len(results) > max_results:
        print(f"  ... and {len(results) - max_results} more. Narrow your keyword.")


def print_activity(act):
    """Print a summary of an activity and its exchanges."""
    print(f"Activity: '{act['name']}'")
    print(f"  Location:          {act['location']}")
    print(f"  Reference product: {act['reference product']}")
    print(f"  Unit:              {act['unit']}")
    print("  Exchanges:")
    for exc in act.exchanges():
        print(f"    [{exc['type']:15s}] {exc.input['name']:<55s} {exc.get('amount', '?'):>10} {exc.input.get('unit', '')}")
    print()


print("Helper functions defined.")

Helper functions defined.


---

## 5. Database initialisation

The existing database (if any) is deleted and recreated from scratch. All activities are created in the cells that follow.

In [14]:
if DB_NAME in bd.databases:
    del bd.databases[DB_NAME]
    print(f"Deleted existing database: {DB_NAME}")

db = bd.Database(DB_NAME)
db.register()
db.write({})  # initialises the underlying SQLite tables
print(f"Created database: {DB_NAME}")

Deleted existing database: lca_database_3DPrintedBiopol
Created database: lca_database_3DPrintedBiopol


---

## 6. Filler production activities

Six filler materials are used in the biopolymer mix. Each follows the same activity structure: raw material input + electricity for milling + machine wear + transport. All are modelled per **1 kg of dry filler powder**.

The fillers are wild-collected or industrial by-products; in all cases the cut-off approach assigns zero upstream burden to the raw material input. Only the processing steps (milling, transport) carry environmental impact.

In Recipe 1 (the reference case), only hemp dust is active (0.199 kg/kg mix). All other fillers are retained as zero-amount placeholders to support optimisation runs with alternative recipes.

### 6.1 Background datasets — fillers

<!-- PLACEHOLDER: search_ei() calls used to verify each raw material dataset, with a note on why each was chosen or why a proxy was used. -->

In [15]:
# ── Retrieve shared background datasets ───────────────────────────────────────
# Datasets used across multiple filler activities.
# search_ei() calls are included as comments to document the verification step.

# search_ei('electricity medium voltage')
ei_electricity = find_ei(
    "market for electricity, medium voltage", "DE",
    ref_product="electricity, medium voltage"
)

# search_ei('industrial machine heavy')
ei_machine = find_ei(
    "market for industrial machine, heavy, unspecified", "RER",
    ref_product="industrial machine, heavy, unspecified"
)

# search_ei('transport freight lorry')
ei_transport = find_ei(
    "market for transport, freight, lorry, unspecified", "RER",
    ref_product="transport, freight, lorry, diesel, unspecified"
)

print("Shared background datasets retrieved.")

Shared background datasets retrieved.


### 6.2 Seagrass filler

Seagrass (*Zostera marina*) washes ashore as wrack and is collected, dried, and milled into powder. The raw material is treated as zero-burden under the cut-off approach: the system boundary starts at collection, and no environmental credit or burden is assigned to the seagrass's growth or beaching.

Drying is modelled as a two-stage process:

1. **Passive air drying (Stage 1):** Beach-cast wrack arrives at approximately 77% moisture content by wet weight (Grokipedia, Beach wrack, 2025) and is spread on racks or ground for passive air/sun drying — a well-established practice in traditional seaweed processing industries (Ireland, Brittany, coastal Norway). This stage requires no electricity input and reduces moisture to approximately 15% wet basis. Solar energy is the drying medium; it is not modelled as an ecoinvent input.

2. **Active finishing drying (Stage 2):** A final mechanical drying step reduces moisture from ~15% to <5% wet basis, suitable for milling into a fine powder for the 3D printing paste. The electricity input for this step is estimated from first principles: removing ~0.1 kg water per kg dry mass at 2,500 kJ/kg latent heat of vaporisation, at 50% dryer thermal efficiency, gives ~0.14 kWh/kg dry. A conservative value of **0.5 kWh/kg dry** is used to account for equipment heat-up losses and real-world inefficiency.

> ⚠️ **Modelling note:** The Pérez-López et al. (2014) reference used in the previous version of this notebook referred to terpene oil extraction from a photobioreactor-cultivated red macroalga (*Ochtodes secundiramea*) — not to *Zostera marina* wrack drying. That citation has been removed. The revised value of 0.5 kWh/kg dry is a first-principles estimate for the finishing step only; it should be updated once the actual drying protocol for the RAW Project is confirmed with CITA partners.
>
> ⚠️ **Sensitivity parameter:** Drying energy range for sensitivity analysis: 0.14–1.0 kWh/kg dry (representing best-case and worst-case finishing step efficiency).

| Flow | Amount | Unit | Source |
|---|---|---|---|
| Wet seagrass (collected wrack) | — | — | Zero burden; not modelled as ecoinvent input |
| Electricity — finishing drying | 0.50 | kWh/kg dry | First-principles estimate; finishing step from ~15% to <5% moisture; ⚠️ confirm with CITA |
| Electricity — milling | 0.35 | kWh/kg dry | Wang et al. (2018), J Wood Sci 64:338–346; proxy from bark flour filler |
| Machine wear | 0.01 | kg/kg dry | Lab-scale estimate; Retsch SM2000 analogy |
| Transport | 0.05 | tkm/kg dry | 50 km × 0.001 t/kg |

In [16]:
# ── Parameters ────────────────────────────────────────────────────────────────
# Two-stage drying model:
#   Stage 1: passive air drying (no electricity) — beach wrack ~77% MC → ~15% MC
#   Stage 2: active finishing drying (electricity) — ~15% MC → <5% MC
#
# Stage 2 first-principles estimate:
#   ~0.1 kg water removed per kg dry mass × 2500 kJ/kg latent heat
#   ÷ 3600 kJ/kWh ÷ 0.50 dryer efficiency = ~0.14 kWh/kg dry
#   Conservative value: 0.5 kWh/kg dry (accounts for equipment heat-up losses)
#   ⚠️ Sensitivity range: 0.14–1.0 kWh/kg dry
#   ⚠️ Update when actual drying protocol confirmed with CITA partners.
DRYING_ENERGY_KWH  = 0.50        # kWh/kg dry; finishing step only
MILLING_ENERGY_KWH = 0.35        # kWh/kg dry; Wang et al. (2018), J Wood Sci 64:338–346
MACHINE_WEAR_KG    = 0.01        # kg machine/kg dry; lab-scale Retsch SM2000 estimate
TRANSPORT_TKM      = 50 * 0.001  # tkm/kg dry; 50 km standard distance

# ── Delete if already exists (safe re-run) ────────────────────────────────────
existing = [a for a in db if a['code'] == 'seagrass_filler_production']
if existing:
    existing[0].delete()
    print("Deleted existing 'seagrass filler production' activity.")

# ── Create activity ───────────────────────────────────────────────────────────
act_seagrass = db.new_activity(
    code="seagrass_filler_production",
    name="seagrass filler production",
    location="GLO",
    unit="kilogram",
    **{"reference product": "seagrass filler",
       "type": "processwithreferenceproduct"}
)
act_seagrass.save()

act_seagrass.new_exchange(input=act_seagrass, amount=1, type="production").save()

# ── Finishing drying electricity ──────────────────────────────────────────────
act_seagrass.new_exchange(
    input=ei_electricity, amount=DRYING_ENERGY_KWH, type="technosphere",
    comment=(
        f"{DRYING_ENERGY_KWH} kWh/kg dry. Two-stage drying model: "
        f"Stage 1 (passive air drying, ~77% → ~15% MC) uses no electricity — "
        f"solar energy only, consistent with traditional seaweed rack-drying practice. "
        f"Stage 2 (active finishing, ~15% → <5% MC): first-principles estimate, "
        f"~0.1 kg water/kg dry × 2500 kJ/kg ÷ 3600 ÷ 0.50 efficiency = 0.14 kWh/kg; "
        f"conservative value of 0.5 kWh/kg used to account for equipment heat-up losses. "
        f"⚠️ Sensitivity range: 0.14–1.0 kWh/kg. "
        f"⚠️ Update when drying protocol confirmed with CITA partners. "
        f"Previous value (4.17 kWh/kg from Pérez-López et al. 2014) removed — "
        f"that reference studied terpene oil extraction from a photobioreactor, "
        f"not Zostera marina wrack drying."
    )
).save()

# ── Milling electricity ───────────────────────────────────────────────────────
act_seagrass.new_exchange(
    input=ei_electricity, amount=MILLING_ENERGY_KWH, type="technosphere",
    comment=(
        f"{MILLING_ENERGY_KWH} kWh/kg dry. Proxy from bark flour filler production "
        f"(Wang et al. 2018, J Wood Sci 64:338–346); same cutting mill type, similar particle size."
    )
).save()

# ── Machine wear ──────────────────────────────────────────────────────────────
act_seagrass.new_exchange(
    input=ei_machine, amount=MACHINE_WEAR_KG, type="technosphere",
    comment=(
        f"{MACHINE_WEAR_KG} kg machine/kg dry. Lab-scale estimate (Retsch SM2000, ~150 kg, "
        f"15-year lifespan, 1,000 h/year, 1 kg/h). Revise to ~1e-4 to 1e-6 kg/kg for industrial LCA."
    )
).save()

# ── Transport ─────────────────────────────────────────────────────────────────
act_seagrass.new_exchange(
    input=ei_transport, amount=TRANSPORT_TKM, type="technosphere",
    comment=f"{TRANSPORT_TKM} tkm/kg dry. Standard 50 km distance used across all filler activities."
).save()

print(f"Created: '{act_seagrass['name']}' — "
      f"{len([e for e in act_seagrass.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Created: 'seagrass filler production' — 4 technosphere inputs


### 6.3 Bark flour filler

Bark flour is produced by milling bark chips into a fine powder. Bark chips are a co-product of sawmill and wood processing operations with an established market (primarily used as wood fuel); upstream burdens from debarking and chipping are therefore allocated to the bark chips and captured in the ecoinvent dataset `market for bark chips, green, measured as dry mass | Europe without Switzerland`. Milling and transport are the additional foreground processes modelled here.

| Flow | Amount | Unit | Source |
|---|---|---|---|
| Bark chips, green (dry mass) | 1.00 | kg/kg | ecoinvent 3.12, `Europe without Switzerland` |
| Electricity — milling | 0.35 | kWh/kg | Wang et al. (2018), J Wood Sci 64:338–346 |
| Machine wear | 0.01 | kg/kg | Lab-scale estimate; Retsch SM2000 analogy |
| Transport | 0.05 | tkm/kg | 50 km × 0.001 t/kg |

In [17]:
# ── Retrieve raw material dataset ─────────────────────────────────────────────
# search_ei('bark chips')
ei_bark = find_ei(
    "market for bark chips, green, measured as dry mass",
    "Europe without Switzerland",
    ref_product="bark chips, green, measured as dry mass"
)

# ── Create activity ───────────────────────────────────────────────────────────
act_bark = db.new_activity(
    code="bark_flour_filler_production",
    name="bark flour filler production",
    location="GLO",
    unit="kilogram",
    **{"reference product": "bark flour filler",
       "type": "processwithreferenceproduct"}
)
act_bark.save()

act_bark.new_exchange(input=act_bark, amount=1, type="production").save()

act_bark.new_exchange(
    input=ei_bark, amount=1.0, type="technosphere",
    comment="Bark chips, green, dry mass basis. ecoinvent 3.12 cut-off, Europe without Switzerland."
).save()

act_bark.new_exchange(
    input=ei_electricity, amount=0.35, type="technosphere",
    comment="Milling electricity: 0.35 kWh/kg. Wang et al. (2018), J Wood Sci 64:338–346."
).save()

act_bark.new_exchange(
    input=ei_machine, amount=0.01, type="technosphere",
    comment=(
        "Machine wear: 0.01 kg/kg. Lab-scale estimate (Retsch SM2000, ~150 kg, "
        "15-year lifespan, 1,000 h/year, 1 kg/h). Revise to ~1e-4 to 1e-6 kg/kg for industrial LCA."
    )
).save()

act_bark.new_exchange(
    input=ei_transport, amount=0.05, type="technosphere",
    comment="50 km × 0.001 t/kg = 0.05 tkm/kg. Standard distance used across all filler activities."
).save()

print(f"Created: '{act_bark['name']}' — {len([e for e in act_bark.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Created: 'bark flour filler production' — 4 technosphere inputs


### 6.4 Cellulose filler

Cellulose filler consists of short fibres rejected during paper production — fibres too short to be incorporated into paper and therefore classified as a process waste stream. Under the cut-off approach, these fibres carry zero upstream burden. Only milling and transport are modelled.

| Flow | Amount | Unit | Source |
|---|---|---|---|
| Cellulose fibres (paper reject fines) | — | — | Zero burden; not modelled as ecoinvent input |
| Electricity — milling | 0.67 | kWh/kg | Wang et al. (2018), J Wood Sci 64:338–346 |
| Machine wear | 0.01 | kg/kg | Lab-scale estimate; Retsch SM2000 analogy |
| Transport | 0.05 | tkm/kg | 50 km × 0.001 t/kg |

Milling electricity is higher than for bark flour (0.67 vs. 0.35 kWh/kg) due to the finer particle size requirements for cellulose fibre.

In [18]:
# ── Create activity ───────────────────────────────────────────────────────────
act_cellulose = db.new_activity(
    code="cellulose_filler_production",
    name="cellulose filler production",
    location="GLO",
    unit="kilogram",
    **{"reference product": "cellulose filler",
       "type": "processwithreferenceproduct"}
)
act_cellulose.save()

act_cellulose.new_exchange(input=act_cellulose, amount=1, type="production").save()

# Note: cellulose fibres (paper reject fines) are a process waste stream with
# zero upstream burden under the cut-off approach. No ecoinvent input is modelled.

act_cellulose.new_exchange(
    input=ei_electricity, amount=0.67, type="technosphere",
    comment=(
        "Milling electricity: 0.67 kWh/kg. Wang et al. (2018), J Wood Sci 64:338–346. "
        "Higher than bark flour (0.35 kWh/kg) due to finer particle size requirements."
    )
).save()

act_cellulose.new_exchange(
    input=ei_machine, amount=0.01, type="technosphere",
    comment=(
        "Machine wear: 0.01 kg/kg. Lab-scale estimate (Retsch SM2000, ~150 kg, "
        "15-year lifespan, 1,000 h/year, 1 kg/h). Revise to ~1e-4 to 1e-6 kg/kg for industrial LCA."
    )
).save()

act_cellulose.new_exchange(
    input=ei_transport, amount=0.05, type="technosphere",
    comment="50 km × 0.001 t/kg = 0.05 tkm/kg. Standard distance used across all filler activities."
).save()

print(f"Created: '{act_cellulose['name']}' — {len([e for e in act_cellulose.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Created: 'cellulose filler production' — 3 technosphere inputs


### 6.5 Cotton filler

Cotton is an agricultural crop carrying the full upstream burden of cultivation and ginning. The raw material input uses the ecoinvent dataset `market for seed-cotton | GLO`, which includes land use, irrigation, pesticide and fertiliser application, and ginning. Milling into filler powder is the additional foreground process modelled here.

| Flow | Amount | Unit | Source |
|---|---|---|---|
| Seed-cotton | 1.00 | kg/kg | ecoinvent 3.12, `GLO` |
| Electricity — milling | 0.212 | kWh/kg | USDA Cotton Ginning Handbook (Funk et al.); range 0.150–0.275 kWh/kg |
| Machine wear | 0.01 | kg/kg | Lab-scale estimate; Retsch SM2000 analogy |
| Transport | 0.05 | tkm/kg | 50 km × 0.001 t/kg |

In [19]:
# ── Retrieve raw material dataset ─────────────────────────────────────────────
# search_ei('seed-cotton')
ei_cotton = find_ei(
    "market for seed-cotton", "GLO",
    ref_product="seed-cotton"
)

# ── Create activity ───────────────────────────────────────────────────────────
act_cotton = db.new_activity(
    code="cotton_filler_production",
    name="cotton filler production",
    location="GLO",
    unit="kilogram",
    **{"reference product": "cotton filler",
       "type": "processwithreferenceproduct"}
)
act_cotton.save()

act_cotton.new_exchange(input=act_cotton, amount=1, type="production").save()

act_cotton.new_exchange(
    input=ei_cotton, amount=1.0, type="technosphere",
    comment="Seed-cotton, GLO. ecoinvent 3.12 cut-off. Full upstream agricultural burden."
).save()

act_cotton.new_exchange(
    input=ei_electricity, amount=0.212, type="technosphere",
    comment=(
        "Milling electricity: 0.212 kWh/kg. USDA Cotton Ginning Handbook (Funk et al.), "
        "ResearchGate ID 365020363. Range 0.150–0.275 kWh/kg; mid-to-upper estimate used."
    )
).save()

act_cotton.new_exchange(
    input=ei_machine, amount=0.01, type="technosphere",
    comment=(
        "Machine wear: 0.01 kg/kg. Lab-scale estimate (Retsch SM2000, ~150 kg, "
        "15-year lifespan, 1,000 h/year, 1 kg/h). Revise to ~1e-4 to 1e-6 kg/kg for industrial LCA."
    )
).save()

act_cotton.new_exchange(
    input=ei_transport, amount=0.05, type="technosphere",
    comment="50 km × 0.001 t/kg = 0.05 tkm/kg. Standard distance used across all filler activities."
).save()

print(f"Created: '{act_cotton['name']}' — {len([e for e in act_cotton.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Created: 'cotton filler production' — 4 technosphere inputs


### 6.6 Hemp dust filler

Hemp dust is the fine particulate residue from hemp fibre processing, with no commercial value and no established market. It is treated as a zero-burden waste stream under the cut-off approach. Milling and transport are the only modelled processes.

Note: the previous version of this activity used `carding waste, hemp | FR` as a proxy raw material input and omitted milling electricity and machine wear. Both are corrected here.

| Flow | Amount | Unit | Source |
|---|---|---|---|
| Hemp dust | — | — | Zero burden; not modelled as ecoinvent input |
| Electricity — milling | 0.35 | kWh/kg | Wang et al. (2018), J Wood Sci 64:338–346; proxy from bark flour |
| Machine wear | 0.01 | kg/kg | Lab-scale estimate; Retsch SM2000 analogy |
| Transport | 0.05 | tkm/kg | 50 km × 0.001 t/kg |

In [20]:
# ── Create activity ───────────────────────────────────────────────────────────
act_hemp = db.new_activity(
    code="hemp_dust_filler_production",
    name="hemp dust filler production",
    location="GLO",
    unit="kilogram",
    **{"reference product": "hemp dust filler",
       "type": "processwithreferenceproduct"}
)
act_hemp.save()

act_hemp.new_exchange(input=act_hemp, amount=1, type="production").save()

# Note: hemp dust is a zero-burden waste stream from hemp fibre processing.
# No ecoinvent raw material input is modelled (cut-off approach).

act_hemp.new_exchange(
    input=ei_electricity, amount=0.35, type="technosphere",
    comment=(
        "Milling electricity: 0.35 kWh/kg. Wang et al. (2018), J Wood Sci 64:338–346; "
        "proxy from bark flour filler (same mill type, similar particle size)."
    )
).save()

act_hemp.new_exchange(
    input=ei_machine, amount=0.01, type="technosphere",
    comment=(
        "Machine wear: 0.01 kg/kg. Lab-scale estimate (Retsch SM2000, ~150 kg, "
        "15-year lifespan, 1,000 h/year, 1 kg/h). Revise to ~1e-4 to 1e-6 kg/kg for industrial LCA."
    )
).save()

act_hemp.new_exchange(
    input=ei_transport, amount=0.05, type="technosphere",
    comment="50 km × 0.001 t/kg = 0.05 tkm/kg. Standard distance used across all filler activities."
).save()

print(f"Created: '{act_hemp['name']}' — {len([e for e in act_hemp.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Created: 'hemp dust filler production' — 3 technosphere inputs


### 6.7 Wood flour filler

Wood flour is produced by milling sawdust into a fine powder. Sawdust is a co-product of sawmill operations with an established market, and upstream sawmill burdens are therefore allocated to it and captured in the ecoinvent dataset `market for sawdust, green, collected, measured as dry mass | Europe without Switzerland`. This is analogous to the treatment of bark chips in Section 6.2. Milling and transport are the additional foreground processes modelled here.

| Flow | Amount | Unit | Source |
|---|---|---|---|
| Sawdust, green (dry mass) | 1.00 | kg/kg | ecoinvent 3.12, `Europe without Switzerland` |
| Electricity — milling | 0.15 | kWh/kg | Wang et al. (2018), J Wood Sci 64:338–346; 250 µm target particle size |
| Machine wear | 0.01 | kg/kg | Lab-scale estimate; Retsch SM2000 analogy |
| Transport | 0.05 | tkm/kg | 50 km × 0.001 t/kg |

Milling electricity is lower than for bark flour (0.15 vs. 0.35 kWh/kg), consistent with the coarser target particle size for wood flour (250 µm).

In [21]:
# ── Retrieve raw material dataset ─────────────────────────────────────────────
# search_ei('sawdust')
ei_sawdust = find_ei(
    "market for sawdust, green, collected, measured as dry mass",
    "Europe without Switzerland",
    ref_product="sawdust, green, collected, measured as dry mass"
)

# ── Create activity ───────────────────────────────────────────────────────────
act_wood = db.new_activity(
    code="wood_flour_filler_production",
    name="wood flour filler production",
    location="GLO",
    unit="kilogram",
    **{"reference product": "wood flour filler",
       "type": "processwithreferenceproduct"}
)
act_wood.save()

act_wood.new_exchange(input=act_wood, amount=1, type="production").save()

act_wood.new_exchange(
    input=ei_sawdust, amount=1.0, type="technosphere",
    comment=(
        "Sawdust, green, dry mass basis. ecoinvent 3.12 cut-off, Europe without Switzerland. "
        "Co-product of sawmill operations; upstream burdens allocated accordingly."
    )
).save()

act_wood.new_exchange(
    input=ei_electricity, amount=0.15, type="technosphere",
    comment=(
        "Milling electricity: 0.15 kWh/kg. Wang et al. (2018), J Wood Sci 64:338–346. "
        "Estimated for 250 µm target particle size (first grinding stage); range 0.10–0.20 kWh/kg."
    )
).save()

act_wood.new_exchange(
    input=ei_machine, amount=0.01, type="technosphere",
    comment=(
        "Machine wear: 0.01 kg/kg. Lab-scale estimate (Retsch SM2000, ~150 kg, "
        "15-year lifespan, 1,000 h/year, 1 kg/h). Revise to ~1e-4 to 1e-6 kg/kg for industrial LCA."
    )
).save()

act_wood.new_exchange(
    input=ei_transport, amount=0.05, type="technosphere",
    comment="50 km × 0.001 t/kg = 0.05 tkm/kg. Standard distance used across all filler activities."
).save()

print(f"Created: '{act_wood['name']}' — {len([e for e in act_wood.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Created: 'wood flour filler production' — 4 technosphere inputs


---

## 7. Binder production chain

The binder chain comprises two foreground activities: pea protein isolate production (AEIEP process) and binder paste production. These are presented together because the isolate has no independent use elsewhere in the database.

### 7.1 AEIEP pea protein isolate production

Pea protein isolate is produced via the Alkaline Extraction / Isoelectric Precipitation (AEIEP) process: whole dried peas are milled, proteins are extracted under alkaline conditions (pH ~8.5), precipitated at the isoelectric point (pH ~4.5), and spray-dried into powder (≥80% protein). No suitable ecoinvent proxy exists for this process; the inventory is built from primary literature sources.

Whole dried peas carry the full upstream agricultural burden (`market for protein pea | GLO`).

| Flow | Amount | Unit | Source |
|---|---|---|---|
| Whole dried peas | 4.00 | kg/kg isolate | GFI/EarthShift (2024); ⚠️ range 3.5–4.5 kg/kg |
| Electricity — milling | 0.08 | kWh/kg | Sanjuán et al. (2014) |
| Electricity — centrifugation | 0.75 | kWh/kg | Berardy et al. (2015); Sanjuán et al. (2014) |
| Electricity — spray drying | 1.60 | kWh/kg | Sanjuán et al. (2014); GFI/EarthShift (2024); ⚠️ range 1.0–2.4 kWh/kg |
| Sodium hydroxide (50% solution) | 0.025 | kg/kg | Berghout et al. (2015); Li et al. (2024) |
| Hydrochloric acid (30% solution) | 0.020 | kg/kg | Berghout et al. (2015); Li et al. (2024) |
| Process water | 15.0 | kg/kg | GFI/EarthShift (2024); ⚠️ range 10–25 kg/kg |
| Machine wear | 0.005 | kg/kg | Conservative estimate for centrifuge + spray dryer |
| Transport | 0.05 | tkm/kg | 50 km × 0.001 t/kg |

In [22]:
# ── Retrieve background datasets ──────────────────────────────────────────────
# search_ei('protein pea')
ei_peas = find_ei("market for protein pea", "GLO", ref_product="protein pea")

# search_ei('sodium hydroxide')
ei_naoh = find_ei(
    "market for sodium hydroxide, without water, in 50% solution state", "RER",
    ref_product="sodium hydroxide, without water, in 50% solution state"
)

# search_ei('hydrochloric acid')
ei_hcl = find_ei(
    "market for hydrochloric acid, without water, in 30% solution state", "RER",
    ref_product="hydrochloric acid, without water, in 30% solution state"
)

# search_ei('tap water')
ei_water = find_ei(
    "market for tap water", "Europe without Switzerland",
    ref_product="tap water"
)

# ── Create activity ───────────────────────────────────────────────────────────
act_aeiep = db.new_activity(
    code="aeiep_pea_protein_isolate_production",
    name="AEIEP pea protein isolate production",
    location="GLO",
    unit="kilogram",
    **{"reference product": "pea protein isolate",
       "type": "processwithreferenceproduct"}
)
act_aeiep.save()

act_aeiep.new_exchange(input=act_aeiep, amount=1, type="production",
    comment="1 kg pea protein isolate (≥80% protein, dry powder) — AEIEP process"
).save()

act_aeiep.new_exchange(
    input=ei_peas, amount=4.0, type="technosphere",
    comment=(
        "Whole dried peas: 4.0 kg/kg isolate. Derived from ~24% protein content and "
        "~67% AEIEP recovery efficiency. Source: GFI/EarthShift (2024). "
        "⚠️ Sensitivity parameter: range 3.5–4.5 kg/kg."
    )
).save()

act_aeiep.new_exchange(
    input=ei_electricity, amount=0.08, type="technosphere",
    comment=(
        "Electricity for pea milling (hammer mill + pin mill): 0.08 kWh/kg isolate. "
        "Sanjuán et al. (2014): legume grinding 0.06–0.10 kWh/kg flour; "
        "applied to 4.0 kg pea input at 0.02 kWh/kg = 0.08 kWh."
    )
).save()

act_aeiep.new_exchange(
    input=ei_electricity, amount=0.75, type="technosphere",
    comment=(
        "Electricity for centrifugation (2 stages): 0.75 kWh/kg isolate. "
        "Berardy et al. (2015) citing Sanjuán et al. (2014): 0.747 kWh/kg."
    )
).save()

act_aeiep.new_exchange(
    input=ei_electricity, amount=1.6, type="technosphere",
    comment=(
        "Electricity for spray drying: 1.6 kWh/kg isolate. Dominant energy step. "
        "Sanjuán et al. (2014); confirmed by GFI/EarthShift (2024). "
        "⚠️ Sensitivity parameter: range 1.0–2.4 kWh/kg (±50%)."
    )
).save()

act_aeiep.new_exchange(
    input=ei_naoh, amount=0.025, type="technosphere",
    comment=(
        "NaOH (50% solution) for alkaline extraction (pH ~8.5): 0.025 kg/kg isolate. "
        "Berghout et al. (2015) via Li et al. (2024): ~40 g/kg for lupin isolate; "
        "25 g/kg adopted for pea AEIEP (milder conditions required)."
    )
).save()

act_aeiep.new_exchange(
    input=ei_hcl, amount=0.02, type="technosphere",
    comment=(
        "HCl (30% solution) for isoelectric precipitation (pH ~4.5): 0.02 kg/kg isolate. "
        "Same source as NaOH. 20 g/kg adopted for pea AEIEP."
    )
).save()

act_aeiep.new_exchange(
    input=ei_water, amount=15.0, type="technosphere",
    comment=(
        "Process water for alkaline extraction and washing: 15.0 kg/kg isolate. "
        "GFI/EarthShift (2024): ~15 kg/kg for industrial wet fractionation. "
        "⚠️ Highest-uncertainty value in this activity. Sensitivity range: 10–25 kg/kg."
    )
).save()

act_aeiep.new_exchange(
    input=ei_machine, amount=0.005, type="technosphere",
    comment=(
        "Machine wear (centrifuge + spray dryer amortised): 0.005 kg/kg isolate. "
        "Conservative estimate for continuous industrial equipment."
    )
).save()

act_aeiep.new_exchange(
    input=ei_transport, amount=0.05, type="technosphere",
    comment="Transport of peas to processing facility: 50 km × 0.001 t/kg = 0.05 tkm."
).save()

print(f"Created: '{act_aeiep['name']}' — {len([e for e in act_aeiep.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Created: 'AEIEP pea protein isolate production' — 9 technosphere inputs


### 7.2 Pea protein binder production

The binder paste is produced by combining pea protein isolate with glycerol (plasticiser), water, and xanthan gum (viscosity modifier), using a hand blender followed by a high-shear dissolver. Amounts are derived from Recipe 1 (Gitsouli protocol, DTU/CITA) and renormalised to the binder fraction (80.1% of total mix).

Xanthan gum has no direct ecoinvent equivalent; carboxymethyl cellulose (CMC) powder is used as a proxy. The amount is negligible (0.0012 kg/kg binder) and the impact is insensitive to proxy choice.

| Flow | Amount | Unit | Source |
|---|---|---|---|
| Pea protein isolate (AEIEP) | 0.1473 | kg/kg binder | Recipe 1, renormalised: 11.8% ÷ 80.1% |
| Glycerol | 0.0375 | kg/kg binder | Recipe 1, renormalised: 3.0% ÷ 80.1% |
| Water | 0.8140 | kg/kg binder | Recipe 1, renormalised: 65.2% ÷ 80.1% |
| Xanthan gum (proxy: CMC powder) | 0.0012 | kg/kg binder | Recipe 1, renormalised: 0.1% ÷ 80.1% |
| Electricity — hand blender | 0.00033 | kWh/kg binder | ~400 W × 5 min; negligible |
| Electricity — dissolver | 0.0029 | kWh/kg binder | ~1500 W × 7 min ÷ 60 kg batch; ⚠️ dissolver model unconfirmed |
| Machine wear — dissolver | 0.00017 | kg/kg binder | 1/6 of dissolver batch time |

In [23]:
# ── Retrieve background datasets ──────────────────────────────────────────────
# search_ei('glycerine')
ei_glycerol = find_ei("market for glycerine", "RER", ref_product="glycerine")

# search_ei('carboxymethyl cellulose')
ei_cmc = find_ei(
    "market for carboxymethyl cellulose, powder", "GLO",
    ref_product="carboxymethyl cellulose, powder"
)

# ── Create activity ───────────────────────────────────────────────────────────
act_binder = db.new_activity(
    code="pea_protein_binder_production",
    name="pea protein binder production",
    location="GLO",
    unit="kilogram",
    **{"reference product": "pea protein binder",
       "type": "processwithreferenceproduct"}
)
act_binder.save()

act_binder.new_exchange(input=act_binder, amount=1, type="production").save()

act_binder.new_exchange(
    input=act_aeiep, amount=0.1473158551810237, type="technosphere",
    comment=(
        "Pea protein isolate: 0.1473 kg/kg binder. "
        "Recipe 1 (11.8% of total mix) renormalised to binder fraction (80.1%): "
        "11.8 / 80.1 = 0.1473 kg. Source: Gitsouli protocol."
    )
).save()

act_binder.new_exchange(
    input=ei_glycerol, amount=0.03745318352059925, type="technosphere",
    comment=(
        "Glycerol (plasticiser): 0.0375 kg/kg binder. "
        "Recipe 1 (3.0% of total mix) renormalised: 3.0 / 80.1 = 0.0375 kg. "
        "Source: Gitsouli protocol."
    )
).save()

act_binder.new_exchange(
    input=ei_water, amount=0.8139825218476905, type="technosphere",
    comment=(
        "Process water: 0.8140 kg/kg binder. "
        "Recipe 1 (65.2% of total mix) renormalised: 65.2 / 80.1 = 0.8140 kg. "
        "No significant evaporation assumed during binder preparation. "
        "Source: Gitsouli protocol."
    )
).save()

act_binder.new_exchange(
    input=ei_cmc, amount=0.001248439450686642, type="technosphere",
    comment=(
        "Xanthan gum (viscosity modifier): 0.0012 kg/kg binder. "
        "Recipe 1 (0.1% of total mix) renormalised: 0.1 / 80.1 = 0.0012 kg. "
        "⚠️ Proxy: CMC powder used as xanthan gum substitute. "
        "Impact negligible at this amount regardless of proxy choice."
    )
).save()

act_binder.new_exchange(
    input=ei_electricity, amount=0.00033, type="technosphere",
    comment=(
        "Hand blender electricity (xanthan gum dissolution): 0.00033 kWh/kg binder. "
        "~400 W × 5 min. Negligible; included for completeness."
    )
).save()

act_binder.new_exchange(
    input=ei_electricity, amount=0.0029, type="technosphere",
    comment=(
        "Dissolver electricity (pea protein homogenisation): 0.0029 kWh/kg binder. "
        "~1500 W × 7 min ÷ 60 kg batch. "
        "⚠️ Dissolver model unconfirmed (CITA). Power rating is an estimate."
    )
).save()

act_binder.new_exchange(
    input=ei_machine, amount=0.0001666666666666667, type="technosphere",
    comment=(
        "Dissolver machine wear (binder step, 1/6 of total batch time): "
        "0.00017 kg/kg binder. ~25 kg machine, 10-year lifespan, 500 h/year, 5 kg/h."
    )
).save()

print(f"Created: '{act_binder['name']}' — {len([e for e in act_binder.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Created: 'pea protein binder production' — 7 technosphere inputs


---

## 8. Mix preparation

Mix preparation combines the binder paste with filler(s) using a high-shear dissolver to produce the final 3D-print-ready biopolymer mix. The activity is modelled per **1 kg of mix**.

Recipe 1 (Gitsouli protocol, DTU/CITA) is the reference case. All non-Recipe-1 fillers are retained as zero-amount placeholder inputs to support optimisation runs with alternative recipes without modifying the activity structure.

| Flow | Amount | Unit | Notes |
|---|---|---|---|
| Pea protein binder | 0.801 | kg/kg mix | Recipe 1: 80.1% binder fraction |
| Hemp dust filler | 0.199 | kg/kg mix | Recipe 1: 19.9% filler fraction |
| Bark flour filler | 0 | kg/kg mix | Placeholder |
| Cellulose filler | 0 | kg/kg mix | Placeholder |
| Cotton filler | 0 | kg/kg mix | Placeholder |
| Seagrass filler | 0 | kg/kg mix | Placeholder |
| Wood flour filler | 0 | kg/kg mix | Placeholder |
| Electricity — dissolver | 0.0125 | kWh/kg mix | ~1500 W × 15 min ÷ 30 kg batch; ⚠️ dissolver model unconfirmed |
| Machine wear — dissolver | 0.00083 | kg/kg mix | 5/6 of dissolver batch time |

In [24]:
# ── Delete if already exists (safe re-run) ────────────────────────────────────
existing = [a for a in db if a['code'] == 'mix_preparation']
if existing:
    existing[0].delete()
    print("Deleted existing 'mix preparation' activity.")

# ── Create activity ───────────────────────────────────────────────────────────
act_mix = db.new_activity(
    code="mix_preparation",
    name="mix preparation",
    location="RER",
    unit="kilogram",
    **{"reference product": "biopolymer mix, 3D-print-ready",
       "type": "processwithreferenceproduct"}
)
act_mix.save()

act_mix.new_exchange(input=act_mix, amount=1, type="production",
    comment="1 kg 3D-print-ready biopolymer mix. Recipe 1 reference case."
).save()

# ── Binder — Recipe 1 active input ───────────────────────────────────────────
act_mix.new_exchange(
    input=act_binder, amount=0.8009999999999999, type="technosphere",
    comment=(
        "Pea protein binder: 0.8010 kg/kg mix (Recipe 1: 80.1% binder fraction). "
        "Source: Gitsouli protocol."
    )
).save()

# ── Fillers — Recipe 1 active input ──────────────────────────────────────────
act_mix.new_exchange(
    input=act_hemp, amount=0.199, type="technosphere",
    comment="Hemp dust filler: 0.199 kg/kg mix. Recipe 1 reference filler. Source: Gitsouli protocol."
).save()

# ── Fillers — zero-amount placeholders for alternative recipes ────────────────
for act, name in [
    (act_bark,      "bark flour filler"),
    (act_cellulose, "cellulose filler"),
    (act_cotton,    "cotton filler"),
    (act_seagrass,  "seagrass filler"),
    (act_wood,      "wood flour filler"),
]:
    act_mix.new_exchange(
        input=act, amount=0, type="technosphere",
        comment=f"Placeholder: 0 kg in Recipe 1. Retained for optimisation runs with other recipes."
    ).save()

# ── Dissolver electricity ─────────────────────────────────────────────────────
act_mix.new_exchange(
    input=ei_electricity, amount=0.0125, type="technosphere",
    comment=(
        "Dissolver electricity (filler homogenisation): 0.0125 kWh/kg mix. "
        "~1500 W × 15 min ÷ 30 kg batch. Protocol: 15–20 min mixing; 15 min used. "
        "⚠️ Dissolver model unconfirmed (CITA). Power rating and batch size are estimates."
    )
).save()

# ── Dissolver machine wear ────────────────────────────────────────────────────
act_mix.new_exchange(
    input=ei_machine, amount=0.0008333333333333334, type="technosphere",
    comment=(
        "Dissolver machine wear (mix step, 5/6 of total batch time): "
        "0.00083 kg/kg mix. ~25 kg machine, 10-year lifespan, 500 h/year, 5 kg/h. "
        "Mix step is the heavier of the two dissolver steps; binder step accounts for 1/6."
    )
).save()

print(f"Created: '{act_mix['name']}' — {len([e for e in act_mix.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Created: 'mix preparation' — 9 technosphere inputs


---

## 9. 3D printing

3D printing is modelled per **1 kg of wet output** (material as it leaves the printer, before drying). Electricity and machine wear are derived from first principles using print rate and power draw data from the CITA robotic printing setup (Carl, DTU/CITA).

Material inputs follow Recipe 1 fractions. All non-Recipe-1 fillers are retained as zero-amount placeholders, consistent with the mix preparation activity.

| Flow | Amount | Unit | Derivation |
|---|---|---|---|
| Mix preparation output | 1.00 | kg/kg wet | Material input = 1 kg wet output (no loss assumed) |
| Electricity | see calculation | kWh/kg wet | 0.4 kW ÷ (4 L/hr × 0.85 kg/L) |
| Machine wear | see calculation | kg/kg wet | 1,000 kg printer ÷ 25-year lifetime ÷ throughput |

**Key assumptions:**
- Print rate: 4 L/hr at typical operating speed (50 mm/s)
- Power draw: 0.4 kW (max 0.58 kW at 2 m/s)
- Mix density: 850 g/L (midpoint of 800–900 g/L range)
- Printer mass: 1,000 kg; lifetime: 25 years continuous operation
- No material loss during printing assumed

In [25]:
# ── Delete if already exists (safe re-run) ────────────────────────────────────
existing = [a for a in db if a['code'] == '3d_printing']
if existing:
    existing[0].delete()
    print("Deleted existing '3D printing' activity.")

# ── First-principles calculation ──────────────────────────────────────────────
# All flows are per kg of wet output (material as it leaves the printer).

# Print rate and density
density_kg_per_L     = 850 / 1000   # kg/L; midpoint of 800–900 g/L
print_rate_L_per_hr  = 4.0          # L/hr at typical operating speed (50 mm/s)
power_kW             = 0.4          # kW; max 0.58 kW at 2 m/s

print_rate_kg_per_hr = print_rate_L_per_hr * density_kg_per_L
hrs_per_kg_wet       = 1 / print_rate_kg_per_hr

# Electricity
kwh_per_kg_wet = power_kW * hrs_per_kg_wet

# Machine wear (allocated by print-hours per kg wet output)
kg_printer        = 1000              # kg; medium industrial robotic arm
hrs_lifetime      = 25 * 365 * 24    # 25-year lifetime in hours
kg_machine_per_kg_wet = (hrs_per_kg_wet / hrs_lifetime) * kg_printer

print(f"Print rate:        {print_rate_kg_per_hr:.3f} kg/hr")
print(f"hrs per kg wet:    {hrs_per_kg_wet:.4f} hrs")
print(f"Electricity:       {kwh_per_kg_wet:.5f} kWh/kg wet")
print(f"Machine wear:      {kg_machine_per_kg_wet:.8f} kg machine/kg wet")

# ── Create activity ───────────────────────────────────────────────────────────
act_printing = db.new_activity(
    code="3d_printing",
    name="3D printing",
    location="RER",
    unit="kilogram",
    **{"reference product": "biopolymer panel, wet",
       "type": "processwithreferenceproduct"}
)
act_printing.save()

act_printing.new_exchange(input=act_printing, amount=1, type="production").save()

# ── Mix input ─────────────────────────────────────────────────────────────────
act_printing.new_exchange(
    input=act_mix, amount=1.0, type="technosphere",
    comment="1 kg mix per kg wet output. No material loss during printing assumed."
).save()

# ── Electricity ───────────────────────────────────────────────────────────────
act_printing.new_exchange(
    input=ei_electricity, amount=kwh_per_kg_wet, type="technosphere",
    comment=(
        f"Electricity: {kwh_per_kg_wet:.5f} kWh/kg wet. "
        f"First-principles: {power_kW} kW × {hrs_per_kg_wet:.4f} hrs/kg. "
        f"Print rate: {print_rate_L_per_hr} L/hr × {density_kg_per_L} kg/L = "
        f"{print_rate_kg_per_hr:.3f} kg/hr. "
        f"Power draw at typical operating speed (50 mm/s); max 0.58 kW at 2 m/s. "
        f"Source: Carl, DTU/CITA."
    )
).save()

# ── Machine wear ──────────────────────────────────────────────────────────────
act_printing.new_exchange(
    input=ei_machine, amount=kg_machine_per_kg_wet, type="technosphere",
    comment=(
        f"Machine wear: {kg_machine_per_kg_wet:.8f} kg/kg wet. "
        f"First-principles: {kg_printer} kg printer, {25}-year lifetime "
        f"({hrs_lifetime:,} hrs), {hrs_per_kg_wet:.4f} hrs/kg wet output. "
        f"Proxy: market for industrial machine, heavy, unspecified | RER."
    )
).save()

print(f"\nCreated: '{act_printing['name']}' — {len([e for e in act_printing.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Print rate:        3.400 kg/hr
hrs per kg wet:    0.2941 hrs
Electricity:       0.11765 kWh/kg wet
Machine wear:      0.00134300 kg machine/kg wet

Created: '3D printing' — 3 technosphere inputs


---

## 10. Drying

Drying is modelled per **1 kg of dry panel output**. The wet panel from 3D printing is dried in a dehumidifying chamber for 2 weeks, removing 75% of the water content.

Energy is calculated from first principles across two contributions:

1. **Dehumidification** — energy to remove water from both infiltrating outdoor air and the wet material, adjusted for dehumidifier COP (2.0).
2. **Temperature control** — energy to heat or cool infiltrating air to maintain chamber temperature, adjusted for heat pump COP (3.0).

Humidity calculations use the explicit Magnus approximation for saturation vapour pressure rather than an external psychrometrics library, keeping the notebook self-contained.

As a sanity check, Carl (DTU/CITA) reported a reference energy consumption of 4 machines × 255 W × 2 weeks. The first-principles result is in the same order of magnitude, giving confidence in the approach.

**Key parameters (from Carl, DTU/CITA unless noted):**

| Parameter | Value | Notes |
|---|---|---|
| Chamber dimensions | 5 × 6 × 3 m | |
| Air infiltration rate | 1 ACH | Typical value for average building |
| Drying time | 14 days (336 hrs) | 2 weeks |
| Chamber RH | 20% | |
| Outdoor RH | 80% | Local climate assumption |
| Temperature (indoor = outdoor) | 20°C | No temperature differential |
| Mix density | 850 g/L | Midpoint of 800–900 g/L range |
| Water content | 682 g/L | Source: Carl |
| Evaporation target | 75% of water content | |
| COP dehumidification | 2.0 | Typical electric dehumidifier |
| COP heating/cooling | 3.0 | Typical heat pump |
| Dehumidifier lifetime | 25 years | |

In [26]:
# ── Delete if already exists (safe re-run) ────────────────────────────────────
existing = [a for a in db if a['code'] == 'drying']
if existing:
    existing[0].delete()
    print("Deleted existing 'Drying' activity.")

# ── Parameters ────────────────────────────────────────────────────────────────
# Local climate
climate_temp_C        = 20       # °C
climate_rh            = 0.80     # relative humidity
atm_pressure_Pa       = 101325   # Pa

# Chamber conditions
chamber_temp_C        = 20       # °C
chamber_rh            = 0.20     # relative humidity
chamber_volume_m3     = 5 * 6 * 3  # m³
air_infiltration_ach  = 1.0      # air changes per hour
drying_time_hrs       = 2 * 7 * 24  # 2 weeks in hours

# Material conditions
n_panels              = 100
liters_per_panel      = 0.5 * 0.5 * 0.05 * 1000  # L
liters_material       = n_panels * liters_per_panel
g_water_per_liter     = 682      # g/L; source: Carl
density_g_per_liter   = 850      # g/L; source: Carl
evaporation_target    = 0.75     # fraction of water to remove

# COP values
cop_dehumidification  = 2.0
cop_heating           = 3.0
cop_cooling           = 3.0

# Machine
hrs_lifetime_dehumidifier = 25 * 365 * 24  # 25-year lifetime in hours

# ── Humidity calculations (Magnus approximation) ──────────────────────────────
# Saturation vapour pressure (Pa) via Magnus formula
def sat_vapour_pressure(T_C):
    return 610.78 * 10 ** (7.5 * T_C / (237.3 + T_C))

# Humidity ratio (kg water / kg dry air) from relative humidity
def humidity_ratio(T_C, rh, P_atm):
    p_sat = sat_vapour_pressure(T_C)
    p_v   = rh * p_sat
    return 0.622 * p_v / (P_atm - p_v)

abs_hum_outdoor  = humidity_ratio(climate_temp_C, climate_rh,  atm_pressure_Pa)
abs_hum_chamber  = humidity_ratio(chamber_temp_C, chamber_rh,  atm_pressure_Pa)
air_density_kg_m3 = 1.2  # kg/m³

# ── Water to remove ───────────────────────────────────────────────────────────
m3_air_total = chamber_volume_m3 * air_infiltration_ach * drying_time_hrs
kg_water_from_air      = (abs_hum_outdoor - abs_hum_chamber) * m3_air_total * air_density_kg_m3
kg_water_in_material   = g_water_per_liter * liters_material / 1000
kg_water_from_material = kg_water_in_material * evaporation_target

# ── Dehumidification energy ───────────────────────────────────────────────────
enthalpy_vap_kJ_per_kg = 2501 - 2.37 * chamber_temp_C  # kJ/kg
kwh_air      = kg_water_from_air      * enthalpy_vap_kJ_per_kg / 3600 / cop_dehumidification
kwh_material = kg_water_from_material * enthalpy_vap_kJ_per_kg / 3600 / cop_dehumidification
kwh_dehum    = kwh_air + kwh_material

# ── Temperature control energy ────────────────────────────────────────────────
kg_air_total       = m3_air_total * air_density_kg_m3
specific_heat_air  = 1.005  # kJ/kg·K
temp_diff          = chamber_temp_C - climate_temp_C
kwh_temp_raw       = kg_air_total * specific_heat_air * abs(temp_diff) / 3600
if temp_diff == 0:
    kwh_temp = 0
elif temp_diff < 0:
    kwh_temp = kwh_temp_raw / cop_cooling
else:
    kwh_temp = kwh_temp_raw / cop_heating

kwh_total_batch = kwh_dehum + kwh_temp

# ── Reference check (Carl's figures) ─────────────────────────────────────────
kwh_carl = 4 * 0.255 * drying_time_hrs  # 4 machines × 255 W × 336 hrs
print(f"First-principles total energy: {kwh_total_batch:.1f} kWh")
print(f"Carl's reference energy:       {kwh_carl:.1f} kWh")
print(f"Ratio (first-principles / Carl): {kwh_total_batch / kwh_carl:.2f}\n")

# ── Normalise to per kg dry output ────────────────────────────────────────────
kg_wet_material      = liters_material * density_g_per_liter / 1000
kg_dry_material      = kg_wet_material - kg_water_from_material
kg_wet_per_kg_dry    = kg_wet_material / kg_dry_material
kwh_per_kg_dry       = kwh_total_batch / kg_dry_material
units_dehum_per_kg_dry = (drying_time_hrs / hrs_lifetime_dehumidifier) / kg_dry_material

print(f"kg wet per kg dry:              {kg_wet_per_kg_dry:.4f}")
print(f"Electricity per kg dry:         {kwh_per_kg_dry:.5f} kWh/kg")
print(f"Dehumidifier wear per kg dry:   {units_dehum_per_kg_dry:.8f} units/kg")

# ── Retrieve dehumidifier dataset ─────────────────────────────────────────────
# search_ei('blower heat exchange')
ei_dehumidifier = find_ei(
    "blower and heat exchange unit production, decentralized, 180-250 m3/h", "RER",
    ref_product="blower and heat exchange unit, decentralized, 180-250 m3/h"
)

# ── Create activity ───────────────────────────────────────────────────────────
act_drying = db.new_activity(
    code="drying",
    name="Drying with dehumidifying chamber",
    location="RER",
    unit="kilogram",
    **{"reference product": "biopolymer panel, dried",
       "type": "processwithreferenceproduct"}
)
act_drying.save()

act_drying.new_exchange(input=act_drying, amount=1, type="production").save()

# Wet panel input (from 3D printing)
act_drying.new_exchange(
    input=act_printing, amount=kg_wet_per_kg_dry, type="technosphere",
    comment=(
        f"Wet panel input: {kg_wet_per_kg_dry:.4f} kg wet per kg dry. "
        f"Accounts for water loss during drying ({evaporation_target*100:.0f}% of water content removed)."
    )
).save()

# Electricity
act_drying.new_exchange(
    input=ei_electricity, amount=kwh_per_kg_dry, type="technosphere",
    comment=(
        f"Drying electricity: {kwh_per_kg_dry:.5f} kWh/kg dry. "
        f"First-principles calculation: dehumidification ({kwh_dehum:.1f} kWh batch) + "
        f"temperature control ({kwh_temp:.1f} kWh batch) = {kwh_total_batch:.1f} kWh total, "
        f"normalised over {kg_dry_material:.1f} kg dry output. "
        f"Sanity check against Carl's reference (4 × 255 W × {drying_time_hrs} hrs = "
        f"{kwh_carl:.1f} kWh): ratio = {kwh_total_batch/kwh_carl:.2f}. "
        f"Chamber: {chamber_volume_m3} m³, {air_infiltration_ach} ACH, {drying_time_hrs} hrs, "
        f"indoor RH {chamber_rh*100:.0f}%, outdoor RH {climate_rh*100:.0f}%, "
        f"COP dehumidification = {cop_dehumidification}."
    )
).save()

# Dehumidifier machine wear
act_drying.new_exchange(
    input=ei_dehumidifier, amount=units_dehum_per_kg_dry, type="technosphere",
    comment=(
        f"Dehumidifier wear: {units_dehum_per_kg_dry:.8f} units/kg dry. "
        f"25-year lifetime ({hrs_lifetime_dehumidifier:,} hrs), "
        f"{drying_time_hrs} hrs per drying cycle, "
        f"{kg_dry_material:.1f} kg dry output per cycle."
    )
).save()

print(f"\nCreated: '{act_drying['name']}' — {len([e for e in act_drying.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

First-principles total energy: 326.9 kWh
Carl's reference energy:       342.7 kWh
Ratio (first-principles / Carl): 0.95

kg wet per kg dry:              2.5111
Electricity per kg dry:         0.77255 kWh/kg
Dehumidifier wear per kg dry:   0.00000363 units/kg

Created: 'Drying with dehumidifying chamber' — 3 technosphere inputs


---

## 11. Kiln baking

Kiln baking is modelled per **1 kg of baked panel output**. The dried panel is baked at 150°C for 45 minutes in a medium studio kiln (~2 m × 1 m × 0.5 m internal volume).

Energy and machine wear are calculated from first principles. Kiln heating elements cycle on/off to maintain temperature, so average power draw (5 kW) is used rather than rated power (7.5 kW), consistent with a typical 50–70% duty cycle for studio kilns.

| Parameter | Value | Source |
|---|---|---|
| Baking duration | 45 min (0.75 hrs) | Gitsouli protocol, DTU/CITA |
| Average power draw | 5 kW | Original notebook estimate; rated power 7.5 kW |
| Panels per kiln load | 3 | Gitsouli protocol |
| Weight per panel (wet) | 3.5 kg | Gitsouli protocol |
| Kiln lifetime | 25 years | Assumption |

⚠️ Sensitivity parameter: power draw range 3.75–5.25 kW (±50% of 5 kW average).

In [27]:
# ── Delete if already exists (safe re-run) ────────────────────────────────────
existing = [a for a in db if a['code'] == 'kiln_baking']
if existing:
    existing[0].delete()
    print("Deleted existing 'Kiln baking' activity.")

# ── Parameters ────────────────────────────────────────────────────────────────
hrs_baking            = 0.75   # hrs; 45 min per Gitsouli protocol (DTU/CITA, June 2026)
                                # note: original notebook had 0 hrs (placeholder)
kw_average_draw       = 5.0    # kW average; rated power 7.5 kW, ~67% duty cycle
                                # ⚠️ Sensitivity range: 3.75–5.25 kW
kg_per_panel          = 3.5    # kg wet panel
n_panels_per_kiln     = 3      # panels per kiln load
hrs_lifetime_kiln     = 25 * 365 * 24  # 25-year lifetime in hours

# ── Calculations ──────────────────────────────────────────────────────────────
kg_wet_per_kiln_load       = n_panels_per_kiln * kg_per_panel
kg_wet_per_kg_baked        = 1.0   # baking assumed to drive off negligible additional mass
                                    # (main water loss occurs during drying step)

kwh_per_batch              = kw_average_draw * hrs_baking
kwh_per_kg_baked           = kwh_per_batch / kg_wet_per_kiln_load * kg_wet_per_kg_baked

hrs_kiln_per_kg_baked      = hrs_baking / kg_wet_per_kiln_load * kg_wet_per_kg_baked
units_kiln_per_kg_baked    = hrs_kiln_per_kg_baked / hrs_lifetime_kiln

print(f"Baking duration:         {hrs_baking} hrs ({hrs_baking*60:.0f} min)")
print(f"kWh per batch:           {kwh_per_batch:.3f} kWh")
print(f"kWh per kg baked:        {kwh_per_kg_baked:.5f} kWh/kg")
print(f"Kiln wear per kg baked:  {units_kiln_per_kg_baked:.10f} units/kg")

# ── Create activity ───────────────────────────────────────────────────────────
act_kiln = db.new_activity(
    code="kiln_baking",
    name="Kiln baking",  # name retained from original database
                                                     # to avoid downstream key conflicts;
                                                     # '2 hours' is incorrect, corrected to 45 min
    location="RER",
    unit="kilogram",
    **{"reference product": "biopolymer panel, baked",
       "type": "processwithreferenceproduct"}
)
act_kiln.save()

act_kiln.new_exchange(input=act_kiln, amount=1, type="production").save()

# Dried panel input
act_kiln.new_exchange(
    input=act_drying, amount=kg_wet_per_kg_baked, type="technosphere",
    comment=(
        f"Dried panel input: {kg_wet_per_kg_baked} kg dried panel per kg baked. "
        f"Baking assumed to drive off negligible additional mass; "
        f"main water loss occurs during the drying step."
    )
).save()

# Electricity
act_kiln.new_exchange(
    input=ei_electricity, amount=kwh_per_kg_baked, type="technosphere",
    comment=(
        f"Kiln electricity: {kwh_per_kg_baked:.5f} kWh/kg baked. "
        f"First-principles: {kw_average_draw} kW average draw × {hrs_baking} hrs "
        f"= {kwh_per_batch} kWh/batch ÷ {kg_wet_per_kiln_load} kg/load. "
        f"Average draw is ~67% of rated power (7.5 kW), consistent with "
        f"heating element cycling (50–70% duty cycle for studio kilns). "
        f"Duration: 45 min per Gitsouli protocol (DTU/CITA, June 2026); "
        f"original notebook had 0 hrs (placeholder). "
        f"⚠️ Sensitivity range: 3.75–5.25 kW average draw."
    )
).save()

# Machine wear
act_kiln.new_exchange(
    input=ei_machine, amount=units_kiln_per_kg_baked, type="technosphere",
    comment=(
        f"Kiln machine wear: {units_kiln_per_kg_baked:.10f} units/kg baked. "
        f"25-year lifetime ({hrs_lifetime_kiln:,} hrs), "
        f"{hrs_baking} hrs per batch, {kg_wet_per_kiln_load} kg per batch. "
        f"Proxy: market for industrial machine, heavy, unspecified | RER."
    )
).save()

print(f"\nCreated: '{act_kiln['name']}' — {len([e for e in act_kiln.exchanges() if e['type'] == 'technosphere'])} technosphere inputs")

Baking duration:         0.75 hrs (45 min)
kWh per batch:           3.750 kWh
kWh per kg baked:        0.35714 kWh/kg
Kiln wear per kg baked:  0.0000003262 units/kg

Created: 'Kiln baking' — 3 technosphere inputs


---

## 12. Full database verification

This section checks the integrity of the completed database: activity count, exchange counts, and linkage through the full process chain.

In [28]:
# ── Activity count and exchange summary ────────────────────────────────────────
print(f"Database: {DB_NAME}")
print(f"Total activities: {len(db)}")
print()
for act in sorted(db, key=lambda x: x['name']):
    tech = [e for e in act.exchanges() if e['type'] == 'technosphere']
    print(f"  {act['name']} ({act['location']}) — {len(tech)} technosphere input(s)")

Database: lca_database_3DPrintedBiopol
Total activities: 12

  3D printing (RER) — 3 technosphere input(s)
  AEIEP pea protein isolate production (GLO) — 9 technosphere input(s)
  Drying with dehumidifying chamber (RER) — 3 technosphere input(s)
  Kiln baking (RER) — 3 technosphere input(s)
  bark flour filler production (GLO) — 4 technosphere input(s)
  cellulose filler production (GLO) — 3 technosphere input(s)
  cotton filler production (GLO) — 4 technosphere input(s)
  hemp dust filler production (GLO) — 3 technosphere input(s)
  mix preparation (RER) — 9 technosphere input(s)
  pea protein binder production (GLO) — 7 technosphere input(s)
  seagrass filler production (GLO) — 4 technosphere input(s)
  wood flour filler production (GLO) — 4 technosphere input(s)


In [29]:
# ── Process chain linkage check ───────────────────────────────────────────────
# Verifies that each step in the main chain links correctly to its upstream input.

chain = [
    ("AEIEP pea protein isolate production", None),
    ("pea protein binder production",        "AEIEP pea protein isolate production"),
    ("mix preparation",                      "pea protein binder production"),
    ("3D printing",                          "mix preparation"),
    ("Drying with dehumidifying chamber",    "3D printing"),
    ("Kiln baking", "Drying with dehumidifying chamber"),
]

print("Process chain linkage check:")
all_ok = True
for act_name, upstream_name in chain:
    act = find_activity(db, act_name)
    if upstream_name is None:
        print(f"  ✓ '{act_name}' — chain start")
        continue
    upstream_links = [
        e for e in act.exchanges()
        if e['type'] == 'technosphere'
        and e.input['name'] == upstream_name
    ]
    if upstream_links:
        amount = upstream_links[0]['amount']
        print(f"  ✓ '{act_name}' ← '{upstream_name}' ({amount:.4f} kg)")
    else:
        print(f"  ✗ '{act_name}' — MISSING link to '{upstream_name}'")
        all_ok = False

print()
if all_ok:
    print("All chain links verified.")
else:
    print("WARNING: one or more chain links are missing — check activities above.")

Process chain linkage check:
  ✓ 'AEIEP pea protein isolate production' — chain start
  ✓ 'pea protein binder production' ← 'AEIEP pea protein isolate production' (0.1473 kg)
  ✓ 'mix preparation' ← 'pea protein binder production' (0.8010 kg)
  ✓ '3D printing' ← 'mix preparation' (1.0000 kg)
  ✓ 'Drying with dehumidifying chamber' ← '3D printing' (2.5111 kg)
  ✓ 'Kiln baking' ← 'Drying with dehumidifying chamber' (1.0000 kg)

All chain links verified.


In [31]:
# ── Test LCA calculation ──────────────────────────────────────────────────────
# Runs GWP100 (EF v3.1) on the reference activity (1 kg baked panel) to confirm
# the database is correctly linked and returns a finite, non-zero result.

method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')
ref    = find_activity(db, "Kiln baking")

lca = bc.LCA({ref: 1}, method)
lca.lci()
lca.lcia()

print(f"Reference activity:  {ref['name']}")
print(f"Functional unit:     1 kg baked biopolymer panel")
print(f"LCIA method:         {method}")
print()
print(f"GWP100:  {lca.score:.4f} kg CO2-eq / kg baked panel")
print()

# Sanity checks
assert lca.score != 0,   "Score is zero — check database linkage"
assert lca.score > 0,    "Score is negative — check for unintended avoided burdens"
assert lca.score < 1000, "Score is implausibly high — check exchange amounts"
print("Sanity checks passed.")

Reference activity:  Kiln baking
Functional unit:     1 kg baked biopolymer panel
LCIA method:         ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

GWP100:  1.9875 kg CO2-eq / kg baked panel

Sanity checks passed.


### Export for Laura

In [51]:
# ── Cell 1: Calculate LCA for final activity (1 kg baked biopolymer panel) ────

import bw2calc as bc
import bw2data as bd
import pandas as pd

# Select all EF v3.1 impact categories (top-level only, no sub-splits)
ef_methods = [m for m in bd.methods if m[0] == 'EF v3.1']
print(f"Found {len(ef_methods)} EF v3.1 impact categories.")

# Final activity: 1 kg of baked biopolymer panel
db  = bd.Database('lca_database_3DPrintedBiopol')
ref = bd.get_node(database='lca_database_3DPrintedBiopol', name='Kiln baking')

functional_units = {'1kg 3D printed biopolymer': {ref.id: 1}}

method_config = {'impact_categories': ef_methods}
data_objs     = bd.get_multilca_data_objs(
    functional_units=functional_units,
    method_config=method_config,
)

mlca = bc.MultiLCA(
    demands      =functional_units,
    method_config=method_config,
    data_objs    =data_objs,
)
mlca.lci()
mlca.lcia()
print("MultiLCA complete.")

df_results = pd.Series(mlca.scores).to_frame(name='score')
print(df_results)

Found 25 EF v3.1 impact categories.
MultiLCA complete.
                                                                                     score
(EF v3.1, acidification, accumulated exceedance... 1kg 3D printed biopolymer  8.438176e-03
(EF v3.1, climate change, global warming potent... 1kg 3D printed biopolymer  1.987489e+00
(EF v3.1, climate change: biogenic, global warm... 1kg 3D printed biopolymer  5.607254e-03
(EF v3.1, climate change: fossil, global warmin... 1kg 3D printed biopolymer  1.974531e+00
(EF v3.1, climate change: land use and land use... 1kg 3D printed biopolymer  7.350585e-03
(EF v3.1, ecotoxicity: freshwater, comparative ... 1kg 3D printed biopolymer  2.011709e+01
(EF v3.1, ecotoxicity: freshwater, inorganics, ... 1kg 3D printed biopolymer  1.256904e+01
(EF v3.1, ecotoxicity: freshwater, organics, co... 1kg 3D printed biopolymer  7.548058e+00
(EF v3.1, energy resources: non-renewable, abio... 1kg 3D printed biopolymer  2.341091e+01
(EF v3.1, eutrophication: freshwate

In [52]:
import os
from datetime import datetime, timezone

EXPORT_DIR = '../results/lca_export'
os.makedirs(EXPORT_DIR, exist_ok=True)
TIMESTAMP  = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

# ── CSV ───────────────────────────────────────────────────────────────────────
CSV_PATH = os.path.join(EXPORT_DIR, f'{TIMESTAMP}_lca_results_EF31.csv')
df_results.to_csv(CSV_PATH)
print(f"CSV saved → {CSV_PATH}")

CSV saved → ../results/lca_export/20260626_095035_lca_results_EF31.csv
